# Subset Selection — Results & Plots

Loads pre-computed subset selection results and generates paper-ready figures comparing selection methods (Fisher information variants, DISCO, random, anchor point, fluid benchmarking) across six safety benchmarks (AirBench 2024, AntRedTeam, HarmBench, SimpleSafety, SafetyBench, WMDP).

**Inputs** — CSV files produced by the subset selection pipeline, expected at:
`data/static_selection_output_data/<benchmark>_<params>_items.csv` and `..._performance.csv`

**Outputs** — PDF figures saved to `Figures/` (created automatically next to this notebook).

**Usage**
1. Run the subset selection pipeline first to populate `data/static_selection_output_data/`.
2. Adjust `BENCHMARK_CONFIG` in the first code cell if you have added benchmarks or changed parameters.
3. Run all cells (`Run All`).

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.special import expit
from scipy import stats
import matplotlib.patches as mpatches
import matplotlib as mpl
import warnings
from scipy.stats import ConstantInputWarning

warnings.filterwarnings("ignore", category=ConstantInputWarning)

FIGURES_DIR = Path("Figures")
FIGURES_DIR.mkdir(exist_ok=True)

BENCHMARK_CONFIG = {
    "helm_airbench2024": {"N_SAMPLES": 70 , "TEST": 10, "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 1000},
    "helm_antredteam":   {"N_SAMPLES": 100, "TEST": 10, "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 500},
    "helm_harmbench":    {"N_SAMPLES": 100, "TEST": 10, "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 400},
    "helm_simplesafety": {"N_SAMPLES": 100, "TEST": 10, "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 100},
    "safetybench":       {"N_SAMPLES": 70 , "TEST": 6 , "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 3000},
    "wmdp":              {"N_SAMPLES": 70 , "TEST": 6 , "N_QUANTILES": 4, "MIN_VARIANCE": 0.01, "STEP": 20, "MAX_ITEMS": 1000},
}

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
INPUT_DATA_PATH = REPO_ROOT / "data/static_selection_output_data"


In [ ]:
# =============================================================================
# Global plot style — paper-ready figures (ICML two-column format)
# Copy this cell into any notebook to get a consistent look across all plots.
# =============================================================================

PALETTE = {
   "blue":           "#0072B2",  # Okabe-Ito blue        — CAT / primary method
   "vermillion":     "#D55E00",  # Okabe-Ito vermillion  — Random / comparison
   "teal":           "#009d9a",  # IBM Teal 50           — third series
   "purple":         "#8a3ffc",  # IBM Purple 50         — fourth series
   "brown":          "#8c564b",  # tab10 brown           — fifth series
   "green":          "#2ca02c",  # tab10 green           — sixth
   "magenta":        "#ee538b",  # IBM Magenta 50        — seventh
   "mauve":          "#CC79A7",  # Okabe-Ito mauve       — eighth
   "sky_blue":       "#56B4E9",  # Okabe-Ito sky blue    — ninth
   "orange":         "#E69F00",  # Okabe-Ito orange      — tenth
   "bluish_green":   "#009E73",  # Okabe-Ito bluish green
   "reddish_purple": "#9467bd",  # tab10 purple
   "black":          "#000000",  # black
   "gray":           "#697077",  # IBM Cool Gray 60      — grid / secondary text
   "darkgray":       "#484C52",
}
# Ordered cycling list — use COLORS[i % len(COLORS)] when iterating benchmarks
COLORS = [v for k, v in PALETTE.items() if k not in ("gray", "darkgray", "black")]


# Semantic shortcuts
COLOR_CAT    = PALETTE["blue"]
COLOR_RANDOM = PALETTE["vermillion"]


# Method colors for self-validity curves
METHOD_COLORS = {
   "MaxFisher+IRT":      PALETTE["blue"],
   "MaxFisher+SafetyScore": PALETTE["teal"],
   "Random+IRT":         PALETTE["vermillion"],
   "Random+SafetyScore":    PALETTE["mauve"],
}
METHOD_LINESTYLES = {
   "MaxFisher+IRT":      "-",
   "MaxFisher+SafetyScore": "--",
   "Random+IRT":         "-",
   "Random+SafetyScore":    "--",
}


SINGLE_COL_WIDTH = 5
DOUBLE_COL_WIDTH = 5.5
HEIGHT = SINGLE_COL_WIDTH/1.62

fig_width = DOUBLE_COL_WIDTH
fig_heigth = HEIGHT


plt.rcParams.update({
   "font.family":        "serif",
   "font.serif":         ["Times New Roman"],
   "font.size":          12,
   "legend.fontsize":    8,
   "lines.linewidth":    1.5,
   "lines.markersize":   4,
   "legend.frameon":     True,
   "legend.framealpha":  0.8,
   "axes.grid":          True,
   "grid.alpha":         0.25,
   "grid.linewidth":     0.5,
   "grid.color":         "#cccccc",
   "axes.facecolor":     "white",
   "figure.facecolor":   "white",
   "figure.dpi":         150,
   "savefig.dpi":        300,
   "savefig.bbox":       "tight",
   "savefig.format":     "pdf",
   "axes.prop_cycle":    mpl.cycler(color=COLORS),
})


In [ ]:
METHODS = [
    "total_fisher",
    "marginal_fisher",
    "marginal_fisher_bquant",
    "disco",
    "random",
    "anchor_point",
    "fluid_benchmarking",
]

COLORS = {
    "total_fisher":           PALETTE["vermillion"],
    "marginal_fisher":        PALETTE["blue"],
    "total_fisher_bquant":    PALETTE["sky_blue"],
    "marginal_fisher_bquant": PALETTE["reddish_purple"],
    "disco":                  PALETTE["orange"],
    "random":                 PALETTE["gray"],
    "anchor_point":           PALETTE["bluish_green"],
    "fluid_benchmarking":     PALETTE["black"],
}

LABELS = {
    "total_fisher":           "Total Fisher",
    "marginal_fisher":        "Marginal Fisher",
    "total_fisher_bquant":    "Total Fisher (b-quantile)",
    "marginal_fisher_bquant": "Marginal Fisher (b-quantile)",
    "disco":                  "DISCO",
    "random":                 "Random",
    "anchor_point":           "Anchor Point",
    "fluid_benchmarking":     "Fluid Bench.",
}


BENCHMARK_NAMES = {
    "helm_harmbench":    "HarmBench",
    "helm_airbench2024": "AIR-Bench 2024",
    "helm_antredteam":   "Anthropic Red Team",
    "helm_simplesafety": "SimpleSafety",
    "safetybench":           "SafetyBench",
    "wmdp":              "WMDP (Bio)",
}

def top_k(params, anchor_b_cols, anchor_b_vals, fold, method, K):
    """Return DataFrame of the top-K items for a method/fold."""
    p = params[params["fold"] == fold].copy()
    if method == "anchor_point":
        if not anchor_b_vals:
            return p.iloc[:0].copy()
        B_snap = min(anchor_b_vals, key=lambda x: abs(x - K))
        return p[p[f"anchor_B_{B_snap}"] == 1].copy()
    rank_col = f"{method}_rank"
    return p.sort_values(rank_col).iloc[:K].copy()

In [ ]:
def extract_matrices_for_fold(fold, max_items, params, subset_perf_df, anchor_b_cols, anchor_b_vals):
    fold_df = subset_perf_df[subset_perf_df["fold"] == fold]

    models = sorted(
        fold_df[(fold_df["method"] == "all_items") & (fold_df["split"] == "test")]["model"].unique()
    )

    def extract_matrix(method, value_col, std_col=None):
        sub = fold_df[
            (fold_df["method"] == method)
            & (fold_df["split"] == "test")
            & (fold_df["K"] <= max_items)
        ]
        if sub.empty or value_col not in sub.columns:
            return None, None
        pivot = sub.pivot(index="model", columns="K", values=value_col)
        mat = pivot.reindex(index=models).sort_index(axis=1).to_numpy()
        if std_col and std_col in sub.columns:
            std_pivot = sub.pivot(index="model", columns="K", values=std_col)
            std_mat = std_pivot.reindex(index=models).sort_index(axis=1).to_numpy()
        else:
            std_mat = np.zeros_like(mat)
        return mat, std_mat

    def extract_k_vals(method):
        sub = fold_df[
            (fold_df["method"] == method)
            & (fold_df["split"] == "test")
            & (fold_df["K"] <= max_items)
        ]
        return sorted(sub["K"].unique())

    def extract_anchor_matrix(value_col, std_col=None):
        sub = fold_df[
            (fold_df["method"] == "anchor_point")
            & (fold_df["split"] == "test")
            & (fold_df["K"] <= max_items)
        ]
        if sub.empty or value_col not in sub.columns:
            return None, None, None
        pivot = sub.pivot(index="model", columns="K", values=value_col)
        k_vals = np.array(sorted(pivot.columns.tolist()))
        mat = pivot.reindex(index=models)[k_vals].to_numpy()
        if std_col and std_col in sub.columns:
            std_pivot = sub.pivot(index="model", columns="K", values=std_col)
            std_mat = std_pivot.reindex(index=models)[k_vals].to_numpy()
        else:
            std_mat = np.zeros_like(mat)
        return mat, std_mat, k_vals

    all_items_test   = fold_df[(fold_df["method"] == "all_items") & (fold_df["split"] == "test")].sort_values("model")
    acc_full         = all_items_test["accuracy"].to_numpy()
    acc_full_std     = all_items_test["accuracy_std"].to_numpy()
    ability_full     = all_items_test["ability"].to_numpy()
    ability_full_std = all_items_test["ability_std"].to_numpy()

    anchor_mat, anchor_std, anchor_k_vals = extract_anchor_matrix("ap_pred")

    return {
        "models":            models,
        "acc_full":          acc_full,
        "acc_full_std":      acc_full_std,
        "ability_full":      ability_full,
        "ability_full_std":  ability_full_std,
        "k_vals":            extract_k_vals("total_fisher"),
        "tf":                extract_matrix("total_fisher",           "ability", "ability_std"),
        "mf":                extract_matrix("marginal_fisher",        "ability", "ability_std"),
        "mf_bquant":         extract_matrix("marginal_fisher_bquant", "ability", "ability_std"),
        "random":            extract_matrix("random",                 "accuracy", "accuracy_std"),
        "disco":             extract_matrix("disco",                  "accuracy"),
        "anchor":            (anchor_mat, anchor_std),
        "anchor_k_vals":     anchor_k_vals,
        "fluid":             extract_matrix("fluid_benchmarking",     "ability", "ability_std"),
        "fluid_k_vals":      extract_k_vals("fluid_benchmarking"),
    }

In [ ]:
# ── Fluid benchmarking: data manipulation ─────────────────────────────────────
# All fluid-specific metric computation lives here.
# add_fluid_metrics_to_fold() is called once per fold from the main benchmark
# loop (after fold_data is built) to inject "fluid" entries into the shared
# fold_corrs_* / fold_nrmse_* dicts consumed by the plotting loops.
# Fluid predicts ability (like IRT methods), so NRMSE and self-consistent ρ
# both use ability_full as the reference.

def add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse):
    """Compute fluid benchmarking correlation and NRMSE for one fold; inject into shared metric dicts."""
    d = fold_data[fold]
    mat, _ = d["fluid"]
    if mat is None:
        return  # no fluid data present for this fold

    acc_full = d["acc_full"]
    ab_full  = d["ability_full"]
    std_ab   = ab_full.std()

    n     = mat.shape[1]
    r_acc = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
    r_ab  = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
    nrmse = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab

    fold_corrs_acc    [fold]["fluid"] = (r_acc, r_acc, r_acc)
    fold_corrs_ability[fold]["fluid"] = (r_ab,  r_ab,  r_ab)
    fold_corrs_sc     [fold]["fluid"] = (r_ab,  r_ab,  r_ab)
    fold_nrmse        [fold]["fluid"] = (nrmse, nrmse, nrmse)

In [ ]:
def average_across_folds(fold_metrics, method_key, folds):
    means = np.array([fold_metrics[f][method_key][0] for f in folds])
    n_valid = np.sum(~np.isnan(means), axis=0)
    avg = np.nanmean(means, axis=0)
    std = np.where(
        n_valid <= 1,
        0.0,
        np.nanstd(means, axis=0, ddof=1) / np.sqrt(n_valid),
    )
    return avg, avg - std, avg + std


def average_rmse_across_folds(fold_rmse, method_key, folds):
    means = np.array([fold_rmse[f][method_key][0] for f in folds])
    n_valid = np.sum(~np.isnan(means), axis=0)
    avg = np.nanmean(means, axis=0)
    std = np.where(
        n_valid <= 1,
        0.0,
        np.nanstd(means, axis=0, ddof=1) / np.sqrt(n_valid),
    )
    return avg, avg - std, avg + std


def _plot(ax, x, avg, lo, hi, method_key):
    color = COLORS[method_key]
    ax.plot(x, avg, "-", color=color, label=LABELS[method_key], linewidth=1.2)
    ax.fill_between(x, lo, hi, color=color, alpha=0.15)

In [ ]:
for bench_name, cfg in BENCHMARK_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  Benchmark: {bench_name}")
    print(f"{'='*60}")

    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError as e:
        print(f"  Skipping: {e}")
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS       = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = np.array([]) if anchor_k_arr is None else np.asarray(anchor_k_arr)
    fluid_k_arr  = np.array(fold_data[folds[0]]["fluid_k_vals"])

    # ── Reference line ────────────────────────────────────────────────────────
    hline_corr = np.mean([
        stats.spearmanr(fold_data[f]["ability_full"], fold_data[f]["acc_full"])[0]
        for f in folds
    ])

    # ── Compute correlations (3 types) and normalised RMSE ────────────────────
    # NRMSE = RMSE / std(reference) — puts IRT-ability and accuracy methods on
    # the same dimensionless scale so they can share a single chart.
    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                # random predicts accuracy, so compare against acc_full
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                # IRT methods predict ability, so compare against ab_full
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

    # ── One figure per correlation type ───────────────────────────────────────
    for fold_c, ylabel, subtitle, fname_key in [
        (fold_corrs_acc,     "Spearman's ρ",     "Spearman's ρ vs accuracy (full)", "corr_acc"),
        (fold_corrs_ability, "Spearman's ρ", "Spearman's ρ vs ability (full)",  "corr_ability"),
        (fold_corrs_sc,      "Spearman's ρ", "Self consistent Spearman's ρ", "corr_sc"),
    ]:
        fig, ax = plt.subplots(figsize=(SINGLE_COL_WIDTH, HEIGHT))
        if subtitle != "":
            ax.set_title(f"{bench_name} — {subtitle}")

        for key, method, k_arr in [
            ("fluid",  "fluid_benchmarking", fluid_k_arr),
        ]:
            valid = [f for f in folds if key in fold_c[f]]
            if valid and len(k_arr) > 0:
                avg, lo, hi = average_across_folds(fold_c, key, valid)
                _plot(ax, k_arr, avg, lo, hi, method)


        for key, method in [
            ("mf",        "marginal_fisher"),
            #("random",    "random"),
            #("tf",        "total_fisher"),
            #("mf_bquant", "marginal_fisher_bquant"),
        ]:
            valid = [f for f in folds if key in fold_c[f]]
            if valid:
                avg, lo, hi = average_across_folds(fold_c, key, valid)
                _plot(ax, K_VALS, avg, lo, hi, method)

        for key, method, k_arr in [
            ("disco",  "disco",              np.array(K_VALS)),
            ("anchor", "anchor_point",       anchor_k_arr),
            #("fluid",  "fluid_benchmarking", fluid_k_arr),
        ]:
            valid = [f for f in folds if key in fold_c[f]]
            if valid and len(k_arr) > 0:
                avg, lo, hi = average_across_folds(fold_c, key, valid)
                _plot(ax, k_arr, avg, lo, hi, method)

        for key, method in [
            #("mf",        "marginal_fisher"),
            ("random",    "random"),
            #("tf",        "total_fisher"),
            #("mf_bquant", "marginal_fisher_bquant"),
        ]:
            valid = [f for f in folds if key in fold_c[f]]
            if valid:
                avg, lo, hi = average_across_folds(fold_c, key, valid)
                _plot(ax, K_VALS, avg, lo, hi, method)

        if subtitle == "ρ vs accuracy (full)":
            ax.axhline(hline_corr, color="black", linestyle="--", linewidth=1.0,
                       label="ability (full)", zorder=0)
        ax.set_xlabel("Subset size (k)")
        ax.set_ylabel(ylabel)
        ax.set_xscale("log")
        ax.set_xlim(1, 1000)
        #ax.set_ylim(0.75, 1.01)
        #ax.legend(loc="lower right", fontsize=6)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc="lower right",
            ncol=2,              # 2×N grid → more square
            fontsize=6,
            frameon=True,
            borderaxespad=0.3,
            columnspacing=0.8,
            handletextpad=0.4,
        )
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / f"{bench_name}_{fname_key}.pdf")
        plt.show()

    # # ── NRMSE: all methods on one chart ───────────────────────────────────────
    # fig, ax = plt.subplots(figsize=(SINGLE_COL_WIDTH, HEIGHT))
    # ax.set_title(f"{bench_name} — Normalised RMSE")
    # for key, method, k_arr in [
    #     ("random",    "random",                np.array(K_VALS)),
    #     ("tf",        "total_fisher",           np.array(K_VALS)),
    #     ("mf",        "marginal_fisher",        np.array(K_VALS)),
    #     ("mf_bquant", "marginal_fisher_bquant", np.array(K_VALS)),
    #     ("disco",     "disco",                  np.array(K_VALS)),
    #     ("anchor",    "anchor_point",           anchor_k_arr),
    #     ("fluid",     "fluid_benchmarking",     fluid_k_arr),
    # ]:
    #     valid = [f for f in folds if key in fold_nrmse[f]]
    #     if valid and len(k_arr) > 0:
    #         avg, lo, hi = average_rmse_across_folds(fold_nrmse, key, valid)
    #         _plot(ax, k_arr, avg, lo, hi, method)
    # ax.set_xlabel("Subset size (k)")
    # ax.set_ylabel("NRMSE  (RMSE ÷ std of reference metric)")
    # ax.set_xscale("log")
    # ax.legend(loc="upper right", fontsize=6)
    # plt.tight_layout()
    # fig.savefig(FIGURES_DIR / f"{bench_name}_nrmse.pdf")
    # plt.show()

print("\nAll benchmarks processed.")

In [ ]:
# Per-benchmark: all methods vs full accuracy, plus random vs full ability
for bench_name, cfg in BENCHMARK_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  Benchmark: {bench_name}")
    print(f"{'='*60}")

    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError as e:
        print(f"  Skipping: {e}")
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS       = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = np.array([]) if anchor_k_arr is None else np.asarray(anchor_k_arr)
    fluid_k_arr  = np.array(fold_data[folds[0]]["fluid_k_vals"])

    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}
    fold_corrs_rand_ab  = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

        sub_rand_ab = subset_perf_df[
            (subset_perf_df["fold"] == fold)
            & (subset_perf_df["method"] == "random")
            & (subset_perf_df["split"] == "test")
            & (subset_perf_df["K"] <= MAX_ITEMS)
        ]
        if not sub_rand_ab.empty and "ability" in sub_rand_ab.columns:
            pivot_ra = sub_rand_ab.pivot(index="model", columns="K", values="ability")
            ra_mat   = pivot_ra.reindex(index=fold_data[fold]["models"]).sort_index(axis=1).to_numpy()
            r_ra_acc = np.array([stats.spearmanr(ra_mat[:, j], acc_full)[0] for j in range(ra_mat.shape[1])])
            fold_corrs_rand_ab[fold]["random"] = (r_ra_acc, r_ra_acc, r_ra_acc)
            if not hasattr(fold_corrs_rand_ab, "_k_arr"):
                fold_corrs_rand_ab["_k_arr"] = np.array(sorted(pivot_ra.columns.tolist()))

    fig, ax = plt.subplots(figsize=(SINGLE_COL_WIDTH, HEIGHT))

    for key, method, k_arr in [("fluid", "fluid_benchmarking", fluid_k_arr)]:
        valid = [f for f in folds if key in fold_corrs_acc[f]]
        if valid and len(k_arr) > 0:
            avg, lo, hi = average_across_folds(fold_corrs_acc, key, valid)
            _plot(ax, k_arr, avg, lo, hi, method)

    for key, method in [
        ("tf",        "total_fisher"),
        ("mf",        "marginal_fisher"),
        ("mf_bquant", "marginal_fisher_bquant"),
    ]:
        valid = [f for f in folds if key in fold_corrs_acc[f]]
        if valid:
            avg, lo, hi = average_across_folds(fold_corrs_acc, key, valid)
            _plot(ax, K_VALS, avg, lo, hi, method)

    for key, method, k_arr in [
        ("disco",  "disco",        np.array(K_VALS)),
        ("anchor", "anchor_point", anchor_k_arr),
    ]:
        valid = [f for f in folds if key in fold_corrs_acc[f]]
        if valid and len(k_arr) > 0:
            avg, lo, hi = average_across_folds(fold_corrs_acc, key, valid)
            _plot(ax, k_arr, avg, lo, hi, method)

    valid = [f for f in folds if "random" in fold_corrs_acc[f]]
    if valid:
        avg, lo, hi = average_across_folds(fold_corrs_acc, "random", valid)
        _plot(ax, K_VALS, avg, lo, hi, "random")

    valid_rab = [f for f in folds if "random" in fold_corrs_rand_ab[f]]
    if valid_rab:
        rand_ab_k = fold_corrs_rand_ab.get("_k_arr", np.array(K_VALS))
        avg_rab, lo_rab, hi_rab = average_across_folds(fold_corrs_rand_ab, "random", valid_rab)
        ax.plot(rand_ab_k, avg_rab, "--", color=COLORS["random"], label="Random IRT", linewidth=1.2)
        ax.fill_between(rand_ab_k, lo_rab, hi_rab, color=COLORS["random"], alpha=0.15)

    ax.set_xlabel("Subset size (k)")
    ax.set_ylabel("Spearman's ρ")
    ax.set_xscale("log")
    ax.set_xlim(1, 1000)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        loc="lower right",
        ncol=2,
        fontsize=9,
        frameon=True,
        borderaxespad=0.3,
        columnspacing=0.8,
        handletextpad=0.4,
    )
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"{bench_name}_corr_acc_all.pdf")
    plt.show()

print("\nAll benchmarks processed.")

In [ ]:
# Per-benchmark: fluid (vs acc), random (vs acc, solid), random (vs ability, dashed)
for bench_name, cfg in BENCHMARK_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  Benchmark: {bench_name}")
    print(f"{'='*60}")

    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError as e:
        print(f"  Skipping: {e}")
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS       = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = np.array([]) if anchor_k_arr is None else np.asarray(anchor_k_arr)
    fluid_k_arr  = np.array(fold_data[folds[0]]["fluid_k_vals"])

    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}
    fold_corrs_rand_ab  = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

        sub_rand_ab = subset_perf_df[
            (subset_perf_df["fold"] == fold)
            & (subset_perf_df["method"] == "random")
            & (subset_perf_df["split"] == "test")
            & (subset_perf_df["K"] <= MAX_ITEMS)
        ]
        if not sub_rand_ab.empty and "ability" in sub_rand_ab.columns:
            pivot_ra = sub_rand_ab.pivot(index="model", columns="K", values="ability")
            ra_mat   = pivot_ra.reindex(index=fold_data[fold]["models"]).sort_index(axis=1).to_numpy()
            r_ra_acc = np.array([stats.spearmanr(ra_mat[:, j], acc_full)[0] for j in range(ra_mat.shape[1])])
            fold_corrs_rand_ab[fold]["random"] = (r_ra_acc, r_ra_acc, r_ra_acc)
            if not hasattr(fold_corrs_rand_ab, "_k_arr"):
                fold_corrs_rand_ab["_k_arr"] = np.array(sorted(pivot_ra.columns.tolist()))

    fig, ax = plt.subplots(figsize=(SINGLE_COL_WIDTH, HEIGHT))

    valid = [f for f in folds if "fluid" in fold_corrs_acc[f]]
    if valid and len(fluid_k_arr) > 0:
        avg, lo, hi = average_across_folds(fold_corrs_acc, "fluid", valid)
        _plot(ax, fluid_k_arr, avg, lo, hi, "fluid_benchmarking")

    valid = [f for f in folds if "random" in fold_corrs_acc[f]]
    if valid:
        avg, lo, hi = average_across_folds(fold_corrs_acc, "random", valid)
        _plot(ax, K_VALS, avg, lo, hi, "random")

    valid_rab = [f for f in folds if "random" in fold_corrs_rand_ab[f]]
    if valid_rab:
        rand_ab_k = fold_corrs_rand_ab.get("_k_arr", np.array(K_VALS))
        avg_rab, lo_rab, hi_rab = average_across_folds(fold_corrs_rand_ab, "random", valid_rab)
        ax.plot(rand_ab_k, avg_rab, "--", color=COLORS["random"], label="Random IRT", linewidth=1.2)
        ax.fill_between(rand_ab_k, lo_rab, hi_rab, color=COLORS["random"], alpha=0.15)

    ax.set_xlabel("Subset size (k)")
    ax.set_ylabel("Spearman's \u03c1")
    ax.set_xscale("log")
    ax.set_xlim(1, 1000)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        loc="lower right",
        ncol=2,
        fontsize=6,
        frameon=True,
        borderaxespad=0.3,
        columnspacing=0.8,
        handletextpad=0.4,
    )
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"{bench_name}_corr_diagnostic.pdf")
    plt.show()

print("\nAll benchmarks processed.")

In [ ]:
# Grid overview: fluid (vs acc) + random (vs acc) + random (vs ability, dashed)
n_bench    = len(BENCHMARK_CONFIG)
ncols      = 3
nrows      = (n_bench + ncols - 1) // ncols

golden              = (1 + 5 ** 0.5) / 2
cell_height         = (DOUBLE_COL_WIDTH / 2) / golden
legend_strip_height = 0.45
fig_height          = nrows * cell_height + legend_strip_height

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(DOUBLE_COL_WIDTH, fig_height),
    squeeze=False,
    sharex=True,
    sharey=True,
)

legend_handles: list = []
legend_labels:  list = []

for ax_idx, (bench_name, cfg) in enumerate(BENCHMARK_CONFIG.items()):
    ax = axes[ax_idx // ncols][ax_idx % ncols]

    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, "data missing", ha="center", va="center",
                transform=ax.transAxes, fontsize=7)
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS      = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = np.array([]) if anchor_k_arr is None else np.asarray(anchor_k_arr)
    fluid_k_arr  = np.array(fold_data[folds[0]]["fluid_k_vals"])

    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}
    fold_corrs_rand_ab  = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n     = mat.shape[1]
            r_acc = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab  = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n     = mat.shape[1]
            r_acc = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab  = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc, r_acc, r_acc)
            fold_corrs_ability[fold][key] = (r_ab,  r_ab,  r_ab)
            fold_corrs_sc     [fold][key] = (r_acc, r_acc, r_acc)
            fold_nrmse        [fold][key] = (nrmse, nrmse, nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

        sub_rand_ab = subset_perf_df[
            (subset_perf_df["fold"] == fold)
            & (subset_perf_df["method"] == "random")
            & (subset_perf_df["split"] == "test")
            & (subset_perf_df["K"] <= MAX_ITEMS)
        ]
        if not sub_rand_ab.empty and "ability" in sub_rand_ab.columns:
            pivot_ra = sub_rand_ab.pivot(index="model", columns="K", values="ability")
            ra_mat   = pivot_ra.reindex(index=fold_data[fold]["models"]).sort_index(axis=1).to_numpy()
            r_ra_acc = np.array([stats.spearmanr(ra_mat[:, j], acc_full)[0] for j in range(ra_mat.shape[1])])
            fold_corrs_rand_ab[fold]["random"] = (r_ra_acc, r_ra_acc, r_ra_acc)
            if not hasattr(fold_corrs_rand_ab, "_k_arr"):
                fold_corrs_rand_ab["_k_arr"] = np.array(sorted(pivot_ra.columns.tolist()))

    valid = [f for f in folds if "fluid" in fold_corrs_acc[f]]
    if valid and len(fluid_k_arr) > 0:
        avg, lo, hi = average_across_folds(fold_corrs_acc, "fluid", valid)
        _plot(ax, fluid_k_arr, avg, lo, hi, "fluid_benchmarking")

    valid = [f for f in folds if "random" in fold_corrs_acc[f]]
    if valid:
        avg, lo, hi = average_across_folds(fold_corrs_acc, "random", valid)
        _plot(ax, K_VALS, avg, lo, hi, "random")

    valid_rab = [f for f in folds if "random" in fold_corrs_rand_ab[f]]
    if valid_rab:
        rand_ab_k = fold_corrs_rand_ab.get("_k_arr", np.array(K_VALS))
        avg_rab, lo_rab, hi_rab = average_across_folds(fold_corrs_rand_ab, "random", valid_rab)
        ax.plot(rand_ab_k, avg_rab, "--", color=COLORS["random"], label="Random IRT", linewidth=1.2)
        ax.fill_between(rand_ab_k, lo_rab, hi_rab, color=COLORS["random"], alpha=0.15)

    ax.set_xscale("log")
    ax.set_axisbelow(True)
    ax.set_xlim(right=1000)

    ax.text(
        0.97, 0.03, BENCHMARK_NAMES.get(bench_name, bench_name),
        transform=ax.transAxes,
        fontsize=plt.rcParams["axes.titlesize"],
        va="bottom", ha="right",
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "none", "alpha": 0.7},
    )

    for h, lbl in zip(*ax.get_legend_handles_labels()):
        if lbl not in legend_labels:
            legend_handles.append(h)
            legend_labels.append(lbl)

for ax_idx in range(n_bench, nrows * ncols):
    axes[ax_idx // ncols][ax_idx % ncols].set_visible(False)

fig.supxlabel("Subset size (k)", fontsize=plt.rcParams["axes.labelsize"], y=0.0)
fig.supylabel("Spearman's \u03c1", fontsize=plt.rcParams["axes.labelsize"], x=0.0)

legend_strip_height = 0.50
fig.tight_layout(rect=(0, 0, 1, 1 - legend_strip_height / fig_height), h_pad=0.1, w_pad=0.1)
fig.subplots_adjust(top=1 - legend_strip_height / fig_height, bottom=0.12, left=0.10)

fig.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    ncol=(len(legend_labels) + 1) // 2,
    bbox_to_anchor=(0.5, 1.0),
    frameon=True,
    framealpha=0.8,
    borderpad=0.3,
    handlelength=1.2,
    columnspacing=0.8,
)

fig.savefig(FIGURES_DIR / "all_benchmarks_grid_corr_diagnostic.pdf")
plt.show()

In [ ]:
import collections

# Accumulate fold-averaged Spearman rhos: (method_label, corr_type, k) -> {bench_name: rho}
all_rhos = collections.defaultdict(dict)

KEY_TO_LABEL = {
    "random":      "random",
    "tf":          "total_fisher",
    "mf":          "marginal_fisher",
    "mf_bquant":   "marginal_fisher_bquant",
    "disco":       "disco",
    "anchor":      "anchor_point",
    "fluid":       "fluid_benchmarking",
}

for bench_name, cfg in BENCHMARK_CONFIG.items():
    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError:
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS       = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = [] if anchor_k_arr is None else list(anchor_k_arr)
    fluid_k_arr  = list(fold_data[folds[0]]["fluid_k_vals"])

    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}
    fold_corrs_rand_ab  = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n      = mat.shape[1]
            r_acc  = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab   = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

        sub_rand_ab = subset_perf_df[
            (subset_perf_df["fold"] == fold)
            & (subset_perf_df["method"] == "random")
            & (subset_perf_df["split"] == "test")
            & (subset_perf_df["K"] <= MAX_ITEMS)
        ]
        if not sub_rand_ab.empty and "ability" in sub_rand_ab.columns:
            pivot_ra = sub_rand_ab.pivot(index="model", columns="K", values="ability")
            ra_mat   = pivot_ra.reindex(index=fold_data[fold]["models"]).sort_index(axis=1).to_numpy()
            r_ra_acc = np.array([stats.spearmanr(ra_mat[:, j], acc_full)[0] for j in range(ra_mat.shape[1])])
            fold_corrs_rand_ab[fold]["random"] = (r_ra_acc, r_ra_acc, r_ra_acc)
            if not hasattr(fold_corrs_rand_ab, "_k_arr"):
                fold_corrs_rand_ab["_k_arr"] = np.array(sorted(pivot_ra.columns.tolist()))

    k_map = {
        "random": K_VALS, "tf": K_VALS, "mf": K_VALS, "mf_bquant": K_VALS,
        "disco":  K_VALS, "anchor": anchor_k_arr, "fluid": fluid_k_arr,
    }

    for corr_type, fold_c in [
        ("corr_acc",     fold_corrs_acc),
        ("corr_ability", fold_corrs_ability),
        ("corr_sc",      fold_corrs_sc),
    ]:
        for method_key, method_label in KEY_TO_LABEL.items():
            k_arr = k_map[method_key]
            if not k_arr:
                continue
            valid = [f for f in folds if method_key in fold_c[f]]
            if not valid:
                continue
            avg, _, _ = average_across_folds(fold_c, method_key, valid)
            for k_val, rho in zip(k_arr, avg):
                all_rhos[(method_label, corr_type, k_val)][bench_name] = rho

    # random ability vs acc_full (separate from the main corr_acc entry for random accuracy)
    rand_ab_k = fold_corrs_rand_ab.get("_k_arr", np.array(K_VALS))
    valid_rab = [f for f in folds if "random" in fold_corrs_rand_ab[f]]
    if valid_rab and rand_ab_k.size > 0:
        avg_rab, _, _ = average_across_folds(fold_corrs_rand_ab, "random", valid_rab)
        for k_val, rho in zip(rand_ab_k, avg_rab):
            all_rhos[("random_ability", "corr_acc", k_val)][bench_name] = rho

rows = []
for (method, corr_type, k), bench_rhos in sorted(all_rhos.items()):
    for bench_name, rho in bench_rhos.items():
        rows.append({"method": method, "corr_type": corr_type, "k": k, "benchmark": bench_name, "rho": rho})

df_csv = pd.DataFrame(rows)
out_path = Path("corr_summary.csv")
df_csv.to_csv(out_path, index=False)
print(f"Saved {len(df_csv)} rows to {out_path}")
df_csv.head(10)

In [ ]:
# Grid overview: all methods, correlation vs full accuracy + Random IRT
n_bench    = len(BENCHMARK_CONFIG)
ncols      = 3
nrows      = (n_bench + ncols - 1) // ncols

golden              = (1 + 5 ** 0.5) / 2
cell_height         = (DOUBLE_COL_WIDTH / 2) / golden
legend_strip_height = 0.45
fig_height          = nrows * cell_height + legend_strip_height

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(DOUBLE_COL_WIDTH, fig_height),
    squeeze=False,
    sharex=True,
    sharey=True,
)

legend_handles: list = []
legend_labels:  list = []

for ax_idx, (bench_name, cfg) in enumerate(BENCHMARK_CONFIG.items()):
    ax = axes[ax_idx // ncols][ax_idx % ncols]

    N_SAMPLES    = cfg["N_SAMPLES"]
    TEST         = cfg["TEST"]
    N_QUANTILES  = cfg["N_QUANTILES"]
    MIN_VARIANCE = cfg["MIN_VARIANCE"]
    STEP         = cfg["STEP"]
    MAX_ITEMS    = cfg["MAX_ITEMS"]
    TAG = INPUT_DATA_PATH / f"{bench_name}_{N_SAMPLES}samples_test{TEST}_{N_QUANTILES}bquant_minvar{MIN_VARIANCE}_step{STEP}"

    try:
        params_df      = pd.read_csv(f"{TAG}_items.csv")
        subset_perf_df = pd.read_csv(f"{TAG}_{MAX_ITEMS}items_performance.csv")
    except FileNotFoundError as e:
        ax.text(0.5, 0.5, "data missing", ha="center", va="center",
                transform=ax.transAxes, fontsize=7)
        continue

    anchor_b_cols = sorted(
        [c for c in params_df.columns if c.startswith("anchor_B_")],
        key=lambda c: int(c.split("_")[-1]),
    )
    anchor_b_vals = [int(c.split("_")[-1]) for c in anchor_b_cols]
    folds = sorted(params_df["fold"].unique())

    fold_data = {
        fold: extract_matrices_for_fold(
            fold, MAX_ITEMS, params_df, subset_perf_df, anchor_b_cols, anchor_b_vals
        )
        for fold in folds
    }
    K_VALS       = fold_data[folds[0]]["k_vals"]
    anchor_k_arr = fold_data[folds[0]]["anchor_k_vals"]
    anchor_k_arr = np.array([]) if anchor_k_arr is None else np.asarray(anchor_k_arr)
    fluid_k_arr  = np.array(fold_data[folds[0]]["fluid_k_vals"])

    fold_corrs_acc     = {f: {} for f in folds}
    fold_corrs_ability = {f: {} for f in folds}
    fold_corrs_sc      = {f: {} for f in folds}
    fold_nrmse         = {f: {} for f in folds}
    fold_corrs_rand_ab = {f: {} for f in folds}

    for fold in folds:
        d        = fold_data[fold]
        acc_full = d["acc_full"]
        ab_full  = d["ability_full"]
        std_ab   = ab_full.std()
        std_acc  = acc_full.std()

        for key in ["random", "tf", "mf", "mf_bquant"]:
            mat, _ = d[key]
            if mat is None:
                continue
            n     = mat.shape[1]
            r_acc = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab  = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            if key == "random":
                nrmse  = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
                sc_ref = r_acc
            else:
                nrmse  = np.sqrt(np.mean((mat - ab_full[:, None]) ** 2, axis=0)) / std_ab
                sc_ref = r_ab
            fold_corrs_acc    [fold][key] = (r_acc,  r_acc,  r_acc)
            fold_corrs_ability[fold][key] = (r_ab,   r_ab,   r_ab)
            fold_corrs_sc     [fold][key] = (sc_ref, sc_ref, sc_ref)
            fold_nrmse        [fold][key] = (nrmse,  nrmse,  nrmse)

        for key, mat_key in [("disco", "disco"), ("anchor", "anchor")]:
            mat, _ = d[mat_key]
            if mat is None:
                continue
            n     = mat.shape[1]
            r_acc = np.array([stats.spearmanr(mat[:, j], acc_full)[0] for j in range(n)])
            r_ab  = np.array([stats.spearmanr(mat[:, j], ab_full )[0] for j in range(n)])
            nrmse = np.sqrt(np.mean((mat - acc_full[:, None]) ** 2, axis=0)) / std_acc
            fold_corrs_acc    [fold][key] = (r_acc, r_acc, r_acc)
            fold_corrs_ability[fold][key] = (r_ab,  r_ab,  r_ab)
            fold_corrs_sc     [fold][key] = (r_acc, r_acc, r_acc)
            fold_nrmse        [fold][key] = (nrmse, nrmse, nrmse)

        add_fluid_metrics_to_fold(fold_data, fold, fold_corrs_acc, fold_corrs_ability, fold_corrs_sc, fold_nrmse)

        sub_rand_ab = subset_perf_df[
            (subset_perf_df["fold"] == fold)
            & (subset_perf_df["method"] == "random")
            & (subset_perf_df["split"] == "test")
            & (subset_perf_df["K"] <= MAX_ITEMS)
        ]
        if not sub_rand_ab.empty and "ability" in sub_rand_ab.columns:
            pivot_ra = sub_rand_ab.pivot(index="model", columns="K", values="ability")
            ra_mat   = pivot_ra.reindex(index=fold_data[fold]["models"]).sort_index(axis=1).to_numpy()
            r_ra_acc = np.array([stats.spearmanr(ra_mat[:, j], acc_full)[0] for j in range(ra_mat.shape[1])])
            fold_corrs_rand_ab[fold]["random"] = (r_ra_acc, r_ra_acc, r_ra_acc)
            fold_corrs_rand_ab["_k_arr"] = np.array(sorted(pivot_ra.columns.tolist()))

    fold_c = fold_corrs_acc

    def _k95(k_arr, avg, thr=0.95):
        mask = ~np.isnan(avg)
        ks, vals = np.asarray(k_arr)[mask], avg[mask]
        hit = np.where(vals >= thr)[0]
        return int(ks[hit[0]]) if len(hit) else None

    print(f"\n{BENCHMARK_NAMES.get(bench_name, bench_name)}")

    for key, method, k_arr in [("fluid", "fluid_benchmarking", fluid_k_arr)]:
        valid = [f for f in folds if key in fold_c[f]]
        if valid and len(k_arr) > 0:
            avg, lo, hi = average_across_folds(fold_c, key, valid)
            _plot(ax, k_arr, avg, lo, hi, method)
            print(f"  {LABELS[method]:<35s} k@95%: {_k95(k_arr, avg)}")

    for key, method in [("tf", "total_fisher"), ("mf", "marginal_fisher"), ("mf_bquant", "marginal_fisher_bquant")]:
        valid = [f for f in folds if key in fold_c[f]]
        if valid:
            avg, lo, hi = average_across_folds(fold_c, key, valid)
            _plot(ax, K_VALS, avg, lo, hi, method)
            print(f"  {LABELS[method]:<35s} k@95%: {_k95(K_VALS, avg)}")

    for key, method, k_arr in [
        ("disco",  "disco",        np.array(K_VALS)),
        ("anchor", "anchor_point", anchor_k_arr),
    ]:
        valid = [f for f in folds if key in fold_c[f]]
        if valid and len(k_arr) > 0:
            avg, lo, hi = average_across_folds(fold_c, key, valid)
            _plot(ax, k_arr, avg, lo, hi, method)
            print(f"  {LABELS[method]:<35s} k@95%: {_k95(k_arr, avg)}")

    valid = [f for f in folds if "random" in fold_c[f]]
    if valid:
        avg, lo, hi = average_across_folds(fold_c, "random", valid)
        _plot(ax, K_VALS, avg, lo, hi, "random")
        print(f"  {LABELS['random']:<35s} k@95%: {_k95(K_VALS, avg)}")

    valid_rab = [f for f in folds if "random" in fold_corrs_rand_ab[f]]
    if valid_rab:
        rand_ab_k = fold_corrs_rand_ab.get("_k_arr", np.array(K_VALS))
        avg_rab, lo_rab, hi_rab = average_across_folds(fold_corrs_rand_ab, "random", valid_rab)
        ax.plot(rand_ab_k, avg_rab, "--", color=COLORS["random"], label="Random IRT", linewidth=1.2)
        ax.fill_between(rand_ab_k, lo_rab, hi_rab, color=COLORS["random"], alpha=0.15)
        print(f"  {'Random IRT':<35s} k@95%: {_k95(rand_ab_k, avg_rab)}")

    ax.set_xscale("log")
    ax.set_axisbelow(True)
    ax.set_xlim(left=1,right=1000)

    ax.text(
        0.97, 0.03, BENCHMARK_NAMES.get(bench_name, bench_name),
        transform=ax.transAxes,
        fontsize=8,
        va="bottom", ha="right",
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "none", "alpha": 0.7},
    )

    for h, lbl in zip(*ax.get_legend_handles_labels()):
        if lbl not in legend_labels:
            legend_handles.append(h)
            legend_labels.append(lbl)

for ax_idx in range(n_bench, nrows * ncols):
    axes[ax_idx // ncols][ax_idx % ncols].set_visible(False)

fig.supxlabel("Subset size (k)", fontsize=plt.rcParams["axes.labelsize"], y=0.0)
fig.supylabel("Spearman's ρ", fontsize=plt.rcParams["axes.labelsize"], x=0.0)

legend_strip_height = 0.50
fig.tight_layout(rect=(0, 0, 1, 1 - legend_strip_height / fig_height), h_pad=0.1, w_pad=0.1)
fig.subplots_adjust(top=1 - legend_strip_height / fig_height, bottom=0.12, left=0.10)

fig.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    ncol=(len(legend_labels) + 1) // 2,
    bbox_to_anchor=(0.5, 1.0),
    frameon=True,
    framealpha=0.8,
    borderpad=0.3,
    handlelength=1.2,
    columnspacing=0.8,
)

fig.savefig(FIGURES_DIR / "all_benchmarks_grid_corr_acc.pdf")
plt.show()